In [1]:
from azure.cosmos import CosmosClient
from datetime import datetime, timezone

In [2]:
def upsert_report_statistics(
    payload: dict,
    cosmos_config: dict,
    project_id: str,
):
    """
    Update report_statistics for a project document in Cosmos DB.

    Args:
        payload (dict): report_statistics payload
        cosmos_config (dict): Cosmos DB configuration
        project_id (str): Project ID to update

    cosmos_config format:
    {
        "endpoint": "<cosmos-endpoint>",
        "key": "<cosmos-key>",
        "database_name": "<database-name>",
        "container_name": "<container-name>"
    }
    """

    # Initialize Cosmos client
    client = CosmosClient(cosmos_config["endpoint"], credential=cosmos_config["key"])

    database = client.get_database_client(cosmos_config["database_name"])

    container = database.get_container_client(cosmos_config["container_name"])

    # Ensure ISO timestamps
    payload["report_start_time"] = (
        payload.get("report_start_time") or datetime.now(timezone.utc).isoformat()
    )

    payload["report_end_time"] = (
        payload.get("report_end_time") or datetime.now(timezone.utc).isoformat()
    )

    # Fetch existing document
    query = "SELECT * FROM c WHERE c.project_id = @project_id"

    params = [{"name": "@project_id", "value": project_id}]

    items = list(
        container.query_items(
            query=query, parameters=params, enable_cross_partition_query=True
        )
    )

    if not items:
        raise Exception(f"Project document not found: {project_id}")

    doc = items[0]

    # Update statistics
    doc["report_statistics"] = payload

    # Optional update timestamp
    doc["updated_on"] = datetime.now(timezone.utc).isoformat()

    # Upsert document
    updated_doc = container.upsert_item(doc)

    return {
        "status": "success",
        "project_id": updated_doc["project_id"],
        "report_statistics": updated_doc["report_statistics"],
    }

In [5]:
cosmos_config = {
    "endpoint": "https://csdb-intertek-esus-dev.documents.azure.com:443/",
    "key": "azcUeVxFxoYoFkChvWI8Wr8lMijOuWXDYQsvMf6O2LmT0Uv3Zs7lDPiXSxWYOjq00MFDbK88ApotACDbODLFXA==",
    "database_name": "intertek_pocplus_dev",
    "container_name": "Project_CDR",
}

payload = {
    "total_llm_calls": 525,
    "total_prompt_tokens": 773632,
    "total_completion_tokens": 9711,
    "total_tokens": 783347,
    "total_files": 16,
    "time_taken": 1542.2408247999992
}

project_id = "L105581614"

In [6]:
res = upsert_report_statistics(
    payload=payload,
    cosmos_config=cosmos_config,
    project_id=project_id
)
res

{'status': 'success',
 'project_id': 'L105581614',
 'report_statistics': {'total_llm_calls': 525,
  'total_prompt_tokens': 773632,
  'total_completion_tokens': 9711,
  'total_tokens': 783347,
  'total_files': 16,
  'time_taken': 1542.2408247999992,
  'report_start_time': '2026-05-25T12:20:42.953958+00:00',
  'report_end_time': '2026-05-25T12:20:42.954000+00:00'}}